# 📓 Notebook 1: LR Baseline – Toxic Comment Detection
**Nhóm Mù Công Nghệ** | Đề tài 22 | Tuần 3–4  
Thành viên: Lê Đỗ Thái Anh (3124410005) · Nhan Thị Ngọc Trân (3124410368) · Nguyễn Công Danh (3124410321)

## 1. Cài đặt thư viện

In [ ]:
# Chạy trên Google Colab hoặc môi trường có GPU
!pip install scikit-learn pandas numpy matplotlib seaborn -q


## 2. Import & cấu hình

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re, os, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    classification_report, hamming_loss,
    roc_auc_score, average_precision_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay
)

LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("✅ Import xong")


## 3. Tải dữ liệu
> Download từ Kaggle: https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge

In [ ]:
# Nếu chạy trên Colab, upload file train.csv lên trước
# from google.colab import files
# files.upload()

df = pd.read_csv('train.csv')
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)


## 4. Khám phá dữ liệu (EDA)

In [ ]:
# Phân bố nhãn
label_counts = df[LABELS].sum().sort_values(ascending=False)
print("Số mẫu dương theo nhãn:")
print(label_counts.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
label_counts.plot(kind='bar', color='steelblue', ax=ax, edgecolor='white')
ax.set_title('Phân bố số mẫu dương theo từng nhãn', fontsize=13, fontweight='bold')
ax.set_xlabel('Nhãn')
ax.set_ylabel('Số mẫu')
ax.tick_params(axis='x', rotation=30)
for i, v in enumerate(label_counts):
    ax.text(i, v + 50, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nTỷ lệ mẫu có ít nhất 1 nhãn: {(df[LABELS].sum(axis=1) > 0).mean():.2%}")


In [ ]:
# Thống kê độ dài văn bản
df['text_len'] = df['comment_text'].str.len()
print(df['text_len'].describe())
df['text_len'].hist(bins=100, figsize=(10,3), color='steelblue', edgecolor='white')
plt.title('Phân bố độ dài bình luận (ký tự)')
plt.xlabel('Số ký tự'); plt.ylabel('Tần suất')
plt.tight_layout(); plt.show()


## 5. Tiền xử lý văn bản

In [ ]:
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)          # loại HTML tags
    text = re.sub(r'http\S+|www\S+', ' ', text)  # loại URL
    text = re.sub(r'[^a-z0-9\s!?.,\'\-]', ' ', text)  # loại ký tự đặc biệt
    text = re.sub(r'\s+', ' ', text).strip()       # chuẩn hóa khoảng trắng
    return text

df['clean_text'] = df['comment_text'].apply(preprocess)

# Kiểm tra
sample_idx = 0
print("BEFORE:", df['comment_text'].iloc[sample_idx][:200])
print("AFTER: ", df['clean_text'].iloc[sample_idx][:200])


## 6. Chia tập train/test
> **QUAN TRỌNG**: chia TRƯỚC khi fit TF-IDF để tránh data leakage

In [ ]:
X = df['clean_text'].values
y = df[LABELS].values  # shape (N, 6) – thứ tự cột khớp với LABELS

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

# Kiểm tra phân bố nhãn giữ nguyên
for i, lbl in enumerate(LABELS):
    train_pos = y_train[:, i].mean()
    test_pos  = y_test[:, i].mean()
    print(f"  {lbl:<15} train={train_pos:.3f}  test={test_pos:.3f}")


## 7. TF-IDF Vectorization

In [ ]:
tfidf = TfidfVectorizer(
    max_features=50_000,
    ngram_range=(1, 2),    # unigram + bigram
    sublinear_tf=True,     # log(1 + tf)
    min_df=3,              # bỏ term quá hiếm
    strip_accents='unicode',
    analyzer='word',
)

# CRITICAL: fit CHỈ trên train
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f"TF-IDF vocab size: {len(tfidf.vocabulary_):,}")
print(f"Train matrix: {X_train_tfidf.shape}")
print(f"Test matrix : {X_test_tfidf.shape}")


## 8. Huấn luyện LR Baseline (threshold = 0.5)

In [ ]:
lr_base = OneVsRestClassifier(
    LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=RANDOM_STATE),
    n_jobs=-1
)
lr_base.fit(X_train_tfidf, y_train)
print("✅ Huấn luyện xong LR Baseline")


In [ ]:
# Dự đoán với threshold mặc định = 0.5
y_pred_base    = lr_base.predict(X_test_tfidf)       # binary (threshold=0.5)
y_proba_base   = lr_base.predict_proba(X_test_tfidf) # xác suất

print("=== CLASSIFICATION REPORT (LR Baseline, threshold=0.5) ===")
print(classification_report(y_test, y_pred_base, target_names=LABELS, digits=3))


## 9. Metric đánh giá đầy đủ

In [ ]:
def evaluate(y_true, y_pred, y_proba, model_name):
    hl   = hamming_loss(y_true, y_pred)
    mif1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    maf1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    roc  = roc_auc_score(y_true, y_proba, average='macro')
    prc  = average_precision_score(y_true, y_proba, average='macro')
    print(f"\n{'='*50}")
    print(f" {model_name}")
    print(f"{'='*50}")
    print(f"  Hamming Loss    : {hl:.4f}")
    print(f"  Micro-F1        : {mif1:.4f}")
    print(f"  Macro-F1        : {maf1:.4f}")
    print(f"  ROC-AUC (macro) : {roc:.4f}")
    print(f"  PR-AUC  (macro) : {prc:.4f}")
    return {'model': model_name, 'hamming': hl, 'micro_f1': mif1,
            'macro_f1': maf1, 'roc_auc': roc, 'pr_auc': prc}

res_base = evaluate(y_test, y_pred_base, y_proba_base, "LR Baseline (threshold=0.5)")


## 10. Confusion Matrix từng nhãn

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, (lbl, ax) in enumerate(zip(LABELS, axes.flatten())):
    cm = confusion_matrix(y_test[:, i], y_pred_base[:, i])
    ConfusionMatrixDisplay(cm, display_labels=['Not', lbl]).plot(
        ax=ax, cmap='Blues', colorbar=False, values_format='d'
    )
    ax.set_title(f'Confusion Matrix – {lbl}', fontweight='bold')
plt.suptitle('LR Baseline – Confusion Matrix 6 nhãn', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('confusion_matrix_baseline.png', dpi=150, bbox_inches='tight')
plt.show()


## 11. Lưu model & kết quả

In [ ]:
import pickle
os.makedirs('results', exist_ok=True)
with open('results/lr_baseline.pkl', 'wb') as f:
    pickle.dump({'model': lr_base, 'tfidf': tfidf}, f)
pd.DataFrame([res_base]).to_csv('results/metrics_baseline.csv', index=False)
print("✅ Đã lưu model vào results/lr_baseline.pkl")
print("✅ Đã lưu metric vào results/metrics_baseline.csv")
